# Does momentum carry through the 1st? — conditioning the day-1 drift (Colab, token-free)

The unconditional day-1 pop is +18.7bp / 61% win. Question: can we make its DIRECTION more predictable by
conditioning on momentum? Five panels:
1. Day-1 (and days 1-3) split by **prior-month UP vs DOWN** — does momentum carry through, or reverse?
2. Day-1 by **prior-month QUINTILE** — is the relationship monotonic?
3. Day-1 by **200-day regime** + **prior-3-month** momentum (slow trend gate).
4. Does a **strong START predict the REST of the month** (days 1-3 vs days 4+)? continuation or fade?
5. The day-of-month curve (1..22) **split by prior-month up/down** — does the whole early-month shape shift?

Run top-to-bottom; last cell exports one file. yfinance→Stooq fallback.


## capture setup


In [ ]:
import builtins as _bi
REPORT_LINES=[]; _realprint=_bi.print
def print(*a, sep=' ', end='\n', **k):
    _realprint(*a, sep=sep, end=end, **k)
    try: REPORT_LINES.append(sep.join(str(x) for x in a))
    except Exception: pass
_realprint('capture ON')


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable,'-m','pip','install','-q',p])
try:
    import pandas as pd, numpy as np
except Exception:
    _pip('pandas'); _pip('numpy'); import pandas as pd, numpy as np
def load_spx():
    try:
        import yfinance as yf
    except Exception:
        _pip('yfinance'); import yfinance as yf
    try:
        df = yf.download('^GSPC', start='1990-01-01', progress=False, auto_adjust=True)
        if len(df):
            s = df['Close']; s = s.iloc[:,0] if hasattr(s,'columns') else s
            return pd.Series(np.asarray(s).ravel(), index=df.index, name='spx').dropna()
    except Exception as e: print('yf failed -> Stooq:', e)
    import urllib.request, io
    raw = urllib.request.urlopen('https://stooq.com/q/d/l/?s=^spx&i=d', timeout=30).read().decode()
    df = pd.read_csv(io.StringIO(raw), parse_dates=['Date']).set_index('Date').sort_index()
    return df['Close'].rename('spx').dropna()
spx = load_spx()
d = pd.DataFrame({'px': spx}); d['ret'] = d['px'].pct_change(); d = d.dropna()
d['ym'] = d.index.to_period('M'); g = d.groupby('ym')
d['dom_start'] = g.cumcount() + 1
d['sma200'] = d['px'].rolling(200).mean(); d['bull'] = d['px'] > d['sma200']
print('SPX', len(d), 'days', d.index[0].date(), '->', d.index[-1].date())


## Build a MONTHLY frame: month return, prior-month, day-1, days1-3, rest-of-month


In [ ]:
mret = d.groupby('ym')['ret'].apply(lambda s:(1+s).prod()-1)
day1 = d[d['dom_start']==1].groupby('ym')['ret'].first()
d13  = d[d['dom_start'].isin([1,2,3])].groupby('ym')['ret'].apply(lambda s:(1+s).prod()-1)
rest = d[d['dom_start']>3].groupby('ym')['ret'].apply(lambda s:(1+s).prod()-1)
bull1 = d[d['dom_start']==1].groupby('ym')['bull'].first()
M = pd.DataFrame({'mret':mret,'day1':day1,'d13':d13,'rest':rest,'bull1':bull1}).dropna(subset=['mret'])
M['prior']  = M['mret'].shift(1)
M['prior3'] = (1+M['mret']).rolling(3).apply(np.prod, raw=True).shift(1) - 1
M = M.dropna(subset=['prior'])
def s(r): return f'n={len(r):4d}  mean {1e4*r.mean():+6.1f}bp  win {100*(r>0).mean():4.0f}%  med {1e4*r.median():+6.1f}bp'
print('monthly rows:', len(M))


## P1 — day-1 & days1-3 split by PRIOR-MONTH up vs down (does momentum carry?)


In [ ]:
up, dn = M['prior']>0, M['prior']<=0
print('DAY-1 after prior month UP  :', s(M.loc[up,'day1']))
print('DAY-1 after prior month DOWN:', s(M.loc[dn,'day1']))
print('DAYS1-3 after prior UP      :', s(M.loc[up,'d13']))
print('DAYS1-3 after prior DOWN    :', s(M.loc[dn,'d13']))
print('\nMomentum CARRIES if UP>DOWN (stronger day-1 after up months); REVERSES if UP<DOWN.')


## P2 — day-1 by PRIOR-MONTH quintile (monotonic?)


In [ ]:
M['q'] = pd.qcut(M['prior'], 5, labels=['Q1 worst','Q2','Q3','Q4','Q5 best'])
for q in ['Q1 worst','Q2','Q3','Q4','Q5 best']:
    print(f'  prior {q:9s} -> day1:', s(M.loc[M['q']==q,'day1']))
print('\nMonotone rising = momentum; falling = reversal; flat = prior month does not predict day-1.')


## P3 — day-1 by 200-day regime and prior-3-month momentum (slow gate)


In [ ]:
print('DAY-1 | price ABOVE 200d (bull):', s(M.loc[M['bull1'],'day1']))
print('DAY-1 | price BELOW 200d (bear):', s(M.loc[~M['bull1'],'day1']))
p3up = M['prior3']>0
print('DAY-1 | prior-3m UP           :', s(M.loc[p3up,'day1']))
print('DAY-1 | prior-3m DOWN         :', s(M.loc[~p3up,'day1']))
# best combo: both slow-bull and prior-month up
combo = (M['prior']>0) & M['bull1']
print('DAY-1 | prior-month UP *and* above-200d:', s(M.loc[combo,'day1']))


## P4 — does a strong START predict the REST of the month? (continuation vs fade)


In [ ]:
c = M['d13'].corr(M['rest'])
print(f'corr(days1-3 return, rest-of-month return) = {c:+.2f}')
hi = M['d13'] > M['d13'].median()
print('REST-of-month after a STRONG start (top-half days1-3):', s(M.loc[hi,'rest']))
print('REST-of-month after a WEAK   start (bot-half days1-3):', s(M.loc[~hi,'rest']))
print('\nPositive corr / stronger-rest-after-strong-start = intra-month momentum (start sets the tone).')
print('Negative = fade (strong starts give it back).')


## P5 — day-of-month curve (1..22) split by prior-month up vs down


In [ ]:
pm = M['prior'].reindex(d['ym'].values); pm.index = d.index    # map prior-month return to each day
d['prior_up'] = (pm.values > 0)
def curve(mask,label):
    t = d[mask].groupby('dom_start')['ret'].mean().reindex(range(1,23))
    print(f'\n{label}:  ' + '  '.join(f'{k}:{1e4*v:+.0f}' for k,v in t.items() if pd.notna(v)))
curve(d['prior_up'],  'after prior month UP   (bp by trading-day-of-month)')
curve(~d['prior_up'], 'after prior month DOWN (bp by trading-day-of-month)')
print('\nSee whether the early-month bid (days1-3) is bigger after UP months (momentum) or DOWN months (reversal/bounce).')


## ⬇️ EXPORT — run LAST


In [ ]:
from datetime import datetime
fname='momentum_through_first_report.txt'
hdr=['='*66,'MOMENTUM-THROUGH-THE-1ST — export',
     f'SPX {d.index[0].date()} -> {d.index[-1].date()} | {len(d)} days | {len(M)} months',
     f'generated {datetime.now().strftime("%Y-%m-%d %H:%M")}','='*66,'']
open(fname,'w').write('\n'.join(hdr)+'\n'+'\n'.join(REPORT_LINES)+'\n')
_realprint('wrote',fname,'—',len(REPORT_LINES),'lines')
if not REPORT_LINES: _realprint('!! empty — run cells above first')
try:
    from google.colab import files; files.download(fname); _realprint('download started')
except Exception as e:
    _realprint('not in Colab / blocked:',e,'- grab it from the folder icon (left)')
